In [ ]:
# === Install Dependencies ===
!pip install gradio transformers cohere torch torchvision pydub ffmpeg-python
!apt-get install ffmpeg
!pip install torch torchvision diffusers transformers accelerate gradio

# === Imports ===
import gradio as gr
import torch
from diffusers import StableDiffusionXLPipeline
import cohere
from PIL import Image, ImageDraw
from transformers import pipeline, BlipProcessor, BlipForConditionalGeneration

# === API Keys ===
HF_TOKEN = "ENTER YOUR HUGGING FACE TOKEN"
COHERE_API_KEY = "ENTER YOUR COHERE API KEY"
co = cohere.Client(COHERE_API_KEY)

# === 1. Text Generation ===
def generate_text(prompt):
    response = co.generate(model="command", prompt=prompt, max_tokens=100)
    return response.generations[0].text.strip()

# === 2. Image Generation with SDXL ===
device = "cuda" if torch.cuda.is_available() else "cpu"
sdxl_pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    use_safetensors=True,
    variant="fp16" if device == "cuda" else None,
    token=HF_TOKEN
).to(device)

def generate_image(prompt):
    image = sdxl_pipe(prompt).images[0]
    return image

# === 3. Speech to Text ===
asr = pipeline("automatic-speech-recognition", model="openai/whisper-base")

def speech_to_text(audio):
    if audio is None:
        return "Please upload an audio file."
    return asr(audio)["text"]

# === 4. Object Detection (Updated) ===
od = pipeline("object-detection", model="hustvl/yolos-small")

def detect_objects(image):
    results = od(image)
    draw = image.copy().convert("RGBA")
    overlay = Image.new("RGBA", draw.size, (255, 255, 255, 0))
    draw_overlay = ImageDraw.Draw(overlay)

    detected_labels = []
    for obj in results:
        box = obj["box"]
        label = obj["label"]
        detected_labels.append(label)

        draw_overlay.rectangle(
            [(box["xmin"], box["ymin"]), (box["xmax"], box["ymax"])],
            outline="red", width=3
        )
        draw_overlay.text((box["xmin"], box["ymin"]), label, fill="red")

    annotated_image = Image.alpha_composite(draw, overlay).convert("RGB")
    object_list_text = ", ".join(set(detected_labels)) if detected_labels else "No objects detected."
    return annotated_image, object_list_text

# === 5. Image Captioning ===
cap_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
cap_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

def caption_image(image):
    inputs = cap_processor(images=image, return_tensors="pt")
    outputs = cap_model.generate(**inputs)
    return cap_processor.decode(outputs[0], skip_special_tokens=True)

# === Gradio UI ===
with gr.Blocks() as demo:
    gr.Markdown("## 🌐 Multimodal AI: Text, Image, Audio, Vision")

    with gr.Tabs():
        with gr.Tab("Text Generation"):
            txt_prompt = gr.Textbox(label="Enter Prompt")
            txt_result = gr.Textbox(label="Generated Text")
            gr.Button("Generate Text").click(generate_text, inputs=txt_prompt, outputs=txt_result)

        with gr.Tab("Image Generation"):
            img_prompt = gr.Textbox(label="Describe the image")
            img_output = gr.Image(label="Generated Image")
            gr.Button("Generate Image").click(generate_image, inputs=img_prompt, outputs=img_output)

        with gr.Tab("Speech to Text"):
            audio_input = gr.Audio(label="Upload Audio", type="filepath")
            audio_output = gr.Textbox(label="Transcribed Text")
            gr.Button("Transcribe").click(speech_to_text, inputs=audio_input, outputs=audio_output)

        with gr.Tab("Object Detection"):
            od_input = gr.Image(label="Upload Image", type="pil")
            od_output_img = gr.Image(label="Detected Objects (Image)")
            od_output_txt = gr.Textbox(label="Detected Object Labels")
            gr.Button("Detect").click(detect_objects, inputs=od_input, outputs=[od_output_img, od_output_txt])

        with gr.Tab("Image Captioning"):
            cap_input = gr.Image(label="Upload Image", type="pil")
            cap_output = gr.Textbox(label="Generated Caption")
            gr.Button("Caption").click(caption_image, inputs=cap_input, outputs=cap_output)

demo.launch()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.6/322.6 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 119.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 94.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 54.5 MB/s eta 0:00:00
   ━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.5/664.8 MB 246.0 MB/s eta 0:00:03